In [15]:
import xmlrpc.client
import json
import base64
import pandas as pd
import numpy as np
import re
from conn_sharepoint import get_auth_token, get_file_from_sharepoint, upload_file_to_sharepoint
from pathlib import Path
import openpyxl
import io
from io import BytesIO
from openpyxl.utils.cell import coordinate_to_tuple
from openpyxl.utils import get_column_letter
import sqlite3
#import schedule
from reportlab.lib.pagesizes import letter, A4, legal
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak, Image
from reportlab.platypus.frames import Frame
from reportlab.platypus.doctemplate import PageTemplate, BaseDocTemplate
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch, cm
from reportlab.lib import colors
from reportlab.lib.colors import HexColor
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY, TA_LEFT, TA_RIGHT
from reportlab.platypus.flowables import HRFlowable

from reportlab.lib.utils import ImageReader
from datetime import datetime
from datetime import date
from zoneinfo import ZoneInfo
from datetime import timezone, timedelta
import load_dotenv
import os
import requests
import time
import pytz
import locale
import tabulate
import traceback
import certifi




In [2]:

class Sharepoint:

    def __init__(self):
        self.token = get_auth_token()

    def download_file(self, file_name, folder_name = ''):
        """
        Descarga un archivo desde una carpeta específica de SharePoint.

        Args:
            file_name (str): Nombre del archivo a descargar.
            folder_name (str): Nombre de la carpeta dentro de la biblioteca de documentos.

        Returns:
            bytes: El contenido del archivo en formato binario si la descarga es exitosa.
            None: Si ocurre un error durante la descarga.
        """

        try:
            # Obtiene la referencia al archivo usando la URL relativa
            file = get_file_from_sharepoint(file_name, self.token)
            # Devuelve el contenido binario del archivo
            return file.content
        
        except Exception as e:
            # Si ocurre un error, lo muestra y devuelve None
            print(f"Error al descargar el archivo {file_name} de SharePoint: {e}")
            return None
    
    def upload_file(self, file_name, content_stream, content_type = None, folder_name = ''):
        """Sube un archivo binario a una carpeta de SharePoint."""
        
        try:
            # Obtiene la referencia de la carpeta destino en SharePoint
        
            content_stream.seek(0)  # Asegura que el stream esté al inicio
            # Sube el archivo usando el contenido del stream
            target_file = upload_file_to_sharepoint(file_name, self.token, content_stream.getvalue(), content_type)

            # Ejecuta la consulta para completar la subida
            
            print(f"\n-> Archivo '{file_name}' subido con éxito en el intento 1")
            return True  # Retorna True si la subida fue exitosa
        
        except Exception as e:
            print(f"Error al subir el archivo {file_name} a SharePoint: {e}")
            return False  # Retorna False si hubo un error durante la subida

    # def upload_file(self, file_name, df):

    #     df = df.applymap(lambda x: x.strftime('%Y-%m-%d %H:%M:%S') if isinstance(x, pd.Timestamp) else x)
    #     df = df.where(pd.notnull(df), None)

    #     print(df.info())

    #     try:
    #         update = update_file_in_sharepoint(file_name, self.token, df)
            
    #         print(update)
    #         #print(f"\n-> Archivo '{file_name}' actualizado con éxito en el intento 1")
    #         return True
    #     except Exception as e:
    #         print(f"Error al actualizar el archivo {file_name} en SharePoint: {e}")
    #         return False
        
        

In [3]:
sp = Sharepoint()

def modify_excel_file(resumen, sheet_name, table_name):
    # Descarga el archivo Excel desde SharePoint
    excel = sp.download_file('https://graph.microsoft.com/v1.0/drives/b!_FsYntHkCECelI9711VeyMbyPx4aLChPm9Ub1jmcZ_XSssm7ywXaR6lOcXtuV77w/root:/Customer%20Experience%2FModelos%20de%20datos%2FModelo%20-%20%20Dashboard%20clientes/Prueba.xlsx:/content')
    if excel:
        try:
            # Convierte los bytes descargados en un objeto BytesIO para manipulación en memoria
            excel_file = io.BytesIO(excel)
            # Carga el archivo Excel en openpyxl
            wl = openpyxl.load_workbook(excel_file)
            # Selecciona la hoja de trabajo especificada
            wh = wl[sheet_name]

            # Obtiene la tabla de la hoja por su nombre
            tabla = wh.tables[table_name]
            # Obtiene la referencia actual de la tabla (ejemplo: 'A1:H10')
            ref_actual = tabla.ref
            # Extrae la coordenada final de la tabla (ejemplo: 'H10')
            coordenada_final = ref_actual.split(':')[-1]
            # Convierte la coordenada final en número de fila y columna
            fila_final_actual, columna_final_num = coordinate_to_tuple(coordenada_final)
            # Calcula la fila donde se insertarán los nuevos datos
            fila_inicio_nuevos_datos = fila_final_actual + 1
                
            # Inserta los nuevos datos fila por fila en la hoja
            for i, fila_nueva in enumerate(resumen):
                for j, valor in enumerate(fila_nueva):
                    wh.cell(row=fila_inicio_nuevos_datos + i, column=j + 1, value=valor)
            
            # Actualiza la referencia de la tabla para incluir las nuevas filas
            fila_final_nueva = fila_final_actual + len(resumen)
            columna_final_letra = get_column_letter(columna_final_num)
            referencia_inicial = ref_actual.split(':')[0]
            nueva_referencia = f'{referencia_inicial}:{columna_final_letra}{fila_final_nueva}'
            tabla.ref = nueva_referencia
            # --- Fin del código de openpyxl ---

            # Guarda el archivo modificado en un nuevo stream de bytes
            excel_stream_out = io.BytesIO()
            wl.save(excel_stream_out)
            excel_stream_out.seek(0)  # Mueve el cursor al inicio del stream
            print(excel_stream_out)
            # # Sube el archivo modificado de vuelta a SharePoint
            # success = sp.upload_file('https://graph.microsoft.com/v1.0/drives/b!_FsYntHkCECelI9711VeyMbyPx4aLChPm9Ub1jmcZ_XSssm7ywXaR6lOcXtuV77w/root:/Customer%20Experience%2FModelos%20de%20datos%2FModelo%20-%20%20Dashboard%20clientes/Prueba.xlsx:/content', excel_stream_out)

        except Exception as e:
            print(f"Error al procesar el archivo Excel: {e}")
            
    else:
        print("No se pudo descargar el archivo.")

def send_data(df, sheet, table):
    manual = df.values.tolist()
    if manual != []:
        modify_excel_file(manual, sheet, table)




In [27]:

pd.set_option('display.max_columns', None)

def ordenar_respuestas(estructura, respuestas):
    #Mapeo de IDs de preguntas a títulos
    question_id_to_title = {q['questionId']: q['title'] for q in estructura['data']['questions']}

    #Lista de DataFrams para almacenar las respuestas
    dfs = []

    #Procesameinto de cada submission
    for submission in respuestas['data']['formSubmissions']: #A nivel de submission
        submission_data = {'#': submission['entryNum'], 'user': submission['submittingUserId']}  #Iniciar un diccionario con el ID de la submission
        #print(f"Submission ID: {submission['formSubmissionId']}")
        for answer in submission.get('answers', []): #A nivel de respuesta
            question_id = answer['questionId']
            question_title = question_id_to_title.get(question_id, f"Pregunta {question_id}") #Usamos el Id para buscar el título de la pregunta

            value = None    
            #Solo considerar aquellas respuestas que no estén vacías o ocultas
            if not answer.get('wasSubmittedEpmty', False) and not answer.get('wasHidden', False):
                question_type = answer.get('questionType', 'unknown')

                #Procesar según el tipo de pregunta
                if question_type == 'openEnded':
                    value = answer.get('value', 'error' )
                elif question_type == 'multipleChoice':
                    selected = [opt['text'] for opt in answer.get('selectedAnswers', [])]
                    value = ', '.join(selected) if selected else 'Ninguna respuesta seleccionada'
                elif question_type == 'yesNo':
                    value = answer.get('selectedIndex', '')
                elif question_type == 'datetime':
                    date_sub = answer.get('timestamp', '')
                    date_sub = answer.get('timestamp', '')  # Obtiene el timestamp
                    dt_utc = datetime.fromtimestamp(date_sub, tz=timezone.utc)  # Convierte a fecha UTC
                    value = dt_utc.strftime("%d/%m/%Y")  # Formatea la fecha
                elif question_type == 'description':
                    value = None
                elif question_type == 'image':
                    value = answer.get('images', '')
                elif question_type == 'signature':
                    value = answer.get('images', '')
                elif question_type == 'rating':
                    value = answer.get('ratingValue', '')
                else:
                    value = 'Tipo no reconocido'
            
            submission_data[question_title] = value
        
        df = pd.DataFrame([submission_data])  #Crear un DataFrame a partir de la respuesta
        dfs.append(df)  #Agregar el DataFrame a la lista
    return dfs

def all_submission(API_key_connecteam):
    url = "https://api.connecteam.com/forms/v1/forms/12914411/form-submissions?limit=100&offset=0"

    headers = {"accept": "application/json",
            "X-API-KEY": f"{API_key_connecteam}"}

    response = requests.get(url, headers=headers)
    response_json = response.json()
    return response_json   


def filter_submissions(API_key_connecteam):
    # Define la zona horaria de Chile
    chile_tz = ZoneInfo("America/Santiago")

    # Obtiene la fecha actual
    today = date(2025, 9, 24)

    # Calcula la fecha de hace 5 días
    yesterday = today - timedelta(days=30)

    # Calcula el inicio del día (medianoche) de hace 5 días en la zona horaria de Chile
    start_of_day_chile = datetime.combine(yesterday, datetime.min.time(), tzinfo=chile_tz)
    # Convierte el inicio del día a UTC
    start_of_day_utc =  start_of_day_chile.astimezone(timezone.utc)
    # Obtiene el timestamp en segundos para el inicio del rango
    start_timestamp_ms = int(start_of_day_utc.timestamp())

    # Calcula el fin del día de mañana en la zona horaria de Chile
    end_of_day_chile = datetime.combine(today + timedelta(days=1), datetime.min.time(), tzinfo=chile_tz)
    # Convierte el fin del día a UTC
    end_of_day_utc = end_of_day_chile.astimezone(timezone.utc)
    # Obtiene el timestamp en segundos para el final del rango, restando 1 para incluir solo hasta el final de hoy
    end_timestamp_ms = int(end_of_day_utc.timestamp()) - 1

    # Construye la URL para la API de Connecteam con los parámetros de rango de fechas y paginación
    url = f"https://api.connecteam.com/forms/v1/forms/12914411/form-submissions?submittingStartTimestamp={start_timestamp_ms}&submittingEndTime={end_timestamp_ms}&limit=100&offset=0"

    # Define los encabezados para la solicitud HTTP, incluyendo la API key
    headers = {
        "accept": "application/json",
        "X-API-KEY": f"{API_key_connecteam}"
    }

    # Realiza la solicitud GET a la API de Connecteam
    response = requests.get(url, headers=headers)
    # Convierte la respuesta en formato JSON
    response_json = response.json()
    # Retorna el JSON con las submissions filtradas por fecha
    return response_json



def form_structure(API_key_connecteam):
    url = "https://api.connecteam.com/forms/v1/forms/12914411"

    headers = {"accept": "application/json",
            "X-API-KEY": f"{API_key_connecteam}"}

    response = requests.get(url, headers=headers)
    response_json = response.json()
    return response_json

def user(API_key_connecteam, user_id):
    url = f"https://api.connecteam.com/users/v1/users?limit=10&offset=0&order=asc&userIds={int(user_id)}&userStatus=active"

    headers = {
        "accept": "application/json",
        "X-API-KEY": f"{API_key_connecteam}"
    }

    response = requests.get(url, headers=headers)

    response_json = response.json()

    nombre_usuario = response_json['data']['users'][0]["firstName"] + " " + response_json['data']['users'][0]["lastName"]
    return nombre_usuario


API_key_c = os.getenv('CONNECTEAM_API_KEY')
ordered_responses = ordenar_respuestas(form_structure(API_key_c), filter_submissions(API_key_c))

lista_capa_1 = []
lista_capa_2 = []

for df in ordered_responses:
    df = df.astype({'user': str})
    
    df_con_datos = df.dropna(axis=1, how ='all') #Eliminando las columnas que no se usaron
    df_columnas = df_con_datos.columns.to_list() #Lista de columnas que si tienen datos

    index_user = df_con_datos.columns.get_loc('user')

    try:
        user_name = user(API_key_c, df_con_datos['user'][0])
    except Exception as e:
        user_name = "Usuario no encontrado"
        print(f"Error al obtener el nombre del usuario: {e}")
        traceback.print_exc()
    
    try:
        df_con_datos.iloc[0, index_user] = user_name # Añadir el nombre del usuario al DataFrame
    except Exception as e:
        print(f"Error al asignar el nombre del usuario al DataFrame: {e}")
        traceback.print_exc()

    #Elementos globales
   

    #Puntos que efectivamente se visitaron
    numeros_visita = set()
    for col in df_columnas:
        # Verificamos si el nombre de la columna comienza con un dígito
        if col and col[0].isdigit():
            # Extraemos el primer carácter (el número)
            numeros_visita.add(col[0])

    numeros_visita = sorted(list(numeros_visita))

    # AQUI INICIA EL ANALISIS POR PUNTO DE MONITOREO VISITADO
    for i in numeros_visita:

        #Separación de los trabajos realizados
        try:
            tipos_realizados = [tipo.strip() for tipo in df_con_datos[f'{i}.2 Tipo de trabajo a realizar'][0].split(',')]
        except Exception as e:
            print(traceback.format_exc())
            tipos_realizados = df_con_datos[f'{i}.2 Tipo de trabajo a realizar']

        # Columnas del punto {1} | general
        columnas_visita = [columna for columna in df_columnas if columna.startswith(i)] #Todas las columnas asociadas al punto i
        #columnas_visita.append(f'{i} Proyecto') 
        columnas_visita = ['#', 'user', 'Fecha visita ', 'Causa visita', 'Nombre del Cliente', 'Calidad del Servicio'] + columnas_visita #Incluyo las columnas generales

        #Dejando un dataframe a nivel de visita de punto
        df_visita = df_con_datos[columnas_visita].copy()


        #Validando si el punto se encuentra seteadao en el listado de connecteam
        if df_visita[f'{i}.1 Punto de monitoreo'][0] == "No encontrado":
        
                #Creamos la columna proyecto
                try:
                    index_columna_punto_proyecto = df_visita.columns.get_loc(f'{i} Proyecto')
                    df_visita.loc[:, f"{i}.1 Proyecto"] = df_visita.iloc[0, index_columna_punto_proyecto]
                    
                    #Definimos el punto ingresado manaualmente como el verdadero
                    index_columna_punto_no = df_visita.columns.get_loc(f'{i}.1 Punto de monitoreo')
                    index_columna_punto_si = df_visita.columns.get_loc(f'{i}.1 Indicar nombre del punto')

                    df_visita.iloc[0, index_columna_punto_no] = df_visita.iloc[0, index_columna_punto_si]

                    del df_visita[f'{i}.1 Indicar nombre del punto']
                except Exception as e:
                    print(f"Error al procesar el punto de monitoreo en OT {df_visita['#'][0]}: {e}")
                    # Si no se encuentra la columna, asignamos un valor por defecto
                    df_visita.loc[:, f"{i}.1 Proyecto"] = "Proyecto no especificado"
                    df_visita.loc[:, f"{i}.1 Punto de monitoreo"] = "Punto no especificado"
                    continue
                    

        else:
            #Buscando el indice de columna
            index_columna_punto = df_visita.columns.get_loc(f'{i}.1 Punto de monitoreo')

            #Definiendo el nombre del proyecto
            match = re.search(r"\[([^\]]*)\]", df_visita.iloc[0, index_columna_punto])
            if match:
                df_visita.loc[:, f"{i}.1 Proyecto"] = match.group(1)
            else:
                df_visita.loc[:, f"{i}.1 Proyecto"] = "Proyecto no especificado"
                            
            #Definiedo el nombre del punto
            # df_visita.iloc[0, index_columna_punto] = re.sub(r"\[[^\]]*\]", "", df_visita.iloc[0, index_columna_punto]).strip() #Eliminando el nombre del proyecto
    
    

        
        id_tipos_realizados = [item.split(' |')[0] for item in tipos_realizados]
        
        #Cantidad de MP realizadas dentro de un mismo punto
        MP_prefijo = set()
        for col in df_visita.columns:
            if ' MP |' in col: # Buscamos ' MP |' para identificar las columnas de MP
                prefix_end_index = col.find(' MP |') + 4 # Sumamos 4 para incluir ' MP'
                prefix = col[:prefix_end_index].strip()
                MP_prefijo.add(prefix)
        
        conteo_instancias_MP = len(MP_prefijo)

        #Cantidad de MC realizadas dentro de un mismo punto
        MC_prefijo = set()
        for col in df_visita.columns:
            if ' MC |' in col: # Buscamos ' MC |' para identificar las columnas de MC
                prefix_end_index = col.find(' MC |') + 4 # Sumamos 4 para incluir ' MC'
                prefix = col[:prefix_end_index].strip()
                MC_prefijo.add(prefix)
        
        conteo_instancias_MC = len(MC_prefijo)

        #Cantidad de I realizadas dentro de un mismo punto
        I_prefijo = set()
        for col in df_visita.columns:
            if ' I |' in col: # Buscamos ' MP |' para identificar las columnas de MP
                # Extraemos el prefijo como '1.2.1 MP' o '1.2.2 MP'
                I_prefix_end_index = col.find(' I |') + 4 # Sumamos 4 para incluir ' MP'
                I_prefix = col[:I_prefix_end_index].strip()
                I_prefijo.add(I_prefix)
        
        conteo_instancias_I = len(I_prefijo)


        # AQUI INICIA EL ANALISIS POR TIPO DE TRABAJO REALIZADO
        for id in id_tipos_realizados:
            #Iniciamos la filtración por tipos de trabajo
            columnas_trabajo = [columna for columna in df_visita.columns if f'{id}' in columna and '|' in columna]
            columnas_trabajo = ['#', 'user', 'Causa visita', f"{i}.1 Proyecto", f"{i}.1 Punto de monitoreo", f"{i}.2 Tipo de trabajo a realizar", 'Fecha visita ', 'Nombre del Cliente'] + columnas_trabajo + [f"{i}.3 Resolución visita", 'Calidad del Servicio']
            df_trabajo = df_visita[columnas_trabajo]
            
            
            def agregar_capa_2(df, id_equipo, id_tipo, lista):
                filtro = f"{i}.2.{id_equipo} {id_tipo}"
                
                try:
                    #Definiendo las columnas para cuando se hacen trabajos a nivel de equipos
                    columnas_equipo = df.filter(like=filtro).columns.to_list()
                    
                    columnas_equipo = ['#', 'user', 'Causa visita', f"{i}.1 Proyecto", f"{i}.1 Punto de monitoreo", f"{i}.2 Tipo de trabajo a realizar", 'Fecha visita ', 'Nombre del Cliente'] + columnas_equipo + [f"{i}.3 Resolución visita", 'Calidad del Servicio']
                except Exception as e:
                    #Al no existir filto, se infiere que son trabajos a nievel de punto
                    columnas_equipo = ['#', 'user', 'Causa visita', f"{i}.1 Proyecto", f"{i}.1 Punto de monitoreo", f"{i}.2 Tipo de trabajo a realizar", 'Fecha visita ', 'Nombre del Cliente', f"{i}.3 Resolución visita", 'Calidad del Servicio']

                df_equipo = df[columnas_equipo]
                df_equipo.loc[:, f"{i}.2 Tipo de trabajo a realizar"] = id_tipo

                lista.append(df_equipo)


            #Tratamiento para Mantención correctiva
            if id == "MC":
                for equipo in range(1, conteo_instancias_MC+1):
                    agregar_capa_2(df_trabajo, equipo, id, lista_capa_2)

            elif id == "MP":
                for equipo in range(1,conteo_instancias_MP+1):
                    agregar_capa_2(df_trabajo, equipo, id, lista_capa_2)

            elif id == "I":
                for equipo in range(1,conteo_instancias_I+1):
                    if f'{i}.2.{equipo} {id} | N° de serie' not in df_trabajo.columns.to_list():
                        index_columna_ns = df_trabajo.columns.get_loc(f'{i}.2.{equipo} {id} | Modelo')
                        df_trabajo.insert(index_columna_ns + 1, f'{i}.2.{equipo} {id} | N° de serie', 'No solicitado')
                        agregar_capa_2(df_trabajo, equipo, id, lista_capa_2)
                    else:
                        agregar_capa_2(df_trabajo, equipo, id, lista_capa_2)
            
            elif id == "ST":
                agregar_capa_2(df_trabajo, '', id, lista_capa_1)
            
            elif id == "LT":
                agregar_capa_2(df_trabajo, '', id, lista_capa_1)
            
            elif id == "G":
                agregar_capa_2(df_trabajo, '', id, lista_capa_1)
            
            elif id == "C":
                agregar_capa_2(df_trabajo, '', id, lista_capa_1)
            

#Creación de los DataFrames finales
d_capa_2 =[ ]
d_capa_1 =[ ]



for df in lista_capa_2:
    fila = df.iloc[0].to_list()
    d_capa_2.append(fila)
    

for df in lista_capa_1:
    fila = df.iloc[0].to_list()
    d_capa_1.append(fila)
    

#Generamos el dataframe final
df_capa_2 = pd.DataFrame(d_capa_2, columns=['OT', 'Técnico','Causa visita', 'Proyecto', 'Asset', 'Tipo de trabajo', 'Fecha visita', 'Cliente', 'Equipo', 'Modelo', 'N° de serie', 'Observaciones', 'Resolución visita', 'Calidad del Servicio'])
df_capa_1 = pd.DataFrame(d_capa_1, columns=['OT', 'Técnico','Causa visita', 'Proyecto', 'Asset', 'Tipo de trabajo', 'Fecha visita', 'Cliente', 'Resolución visita', 'Calidad del Servicio'])
df_capas = pd.concat([df_capa_1, df_capa_2], ignore_index=True)



In [28]:
# Vista previa de los datos procesados
# df_capas[df_capas['Asset'] == '[Global Tórtolas] Pozo PBM5']
df_capas.tail(10)




,OT,Técnico,Causa visita,Proyecto,Asset,Tipo de trabajo,Fecha visita,Cliente,Resolución visita,Calidad del Servicio,Equipo,Modelo,N° de serie,Observaciones
133,122,Juan José López,Integración de sondas y soluciones flowcell,Global Tórtolas,[Global Tórtolas] Pozo O4B,I,11/09/2025,Patricio Cortez,Se realizó la instalación de la celda de flujo...,3,Sonda multiparamétrica,HL4,21299H405136,.
134,122,Juan José López,Integración de sondas y soluciones flowcell,Global Tórtolas,[Global Tórtolas] Pozo PBM4,I,11/09/2025,Patricio Cortez,PBM4 Se realizó la canalización del microtubo ...,3,Sonda multiparamétrica,Kase,…,
135,122,Juan José López,Integración de sondas y soluciones flowcell,Proyecto no especificado,Ninguna respuesta seleccionada,I,11/09/2025,Patricio Cortez,P6 Se realizó la canalización del microtubo ha...,3,Sonda multiparamétrica,…,…,
136,122,Juan José López,Integración de sondas y soluciones flowcell,Proyecto no especificado,Ninguna respuesta seleccionada,I,11/09/2025,Patricio Cortez,"P3G Se realizó la canalización del microtubo, ...",3,Sonda multiparamétrica,…,…,
137,125,Juan José López,Integración sonda multiparametrica,Proyecto no especificado,Ninguna respuesta seleccionada,I,12/09/2025,Patricio Corte,Se realizó la instalación de la celda de flujo...,3,Sonda multiparamétrica,HL4,21350H405173,Se trasladó los equipos del recinto PME-11 par...
138,125,Juan José López,Integración sonda multiparametrica,Proyecto no especificado,Ninguna respuesta seleccionada,I,12/09/2025,Patricio Corte,Se realizó la instalación de la celda de flujo...,3,Sonda multiparamétrica,HL4,20206H404833,Se traslada cable de sonda multiparametrica de...
139,133,Camilo Sandoval,"Integración de sonda, reconocimiento pozo PP31.",Global Tórtolas,[Global Tórtolas] Pozo Dren ME Nodo,I,16/09/2025,Patricio Cortez,"Se lleva a cabo la instalación, configuración ...",3,Sonda multiparamétrica,HL4,..,
140,136,Juan José López,Integración sonda multiparametrica,Global Tórtolas,[Global Tórtolas] Pozo Dren ME Nodo,I,17/09/2025,Patricio Cortez,Se realizó la instalación de la sonda multipar...,3,Sonda multiparamétrica,HL4,21316H405166,La sonda instalada no tiene sensor para tener ...
141,139,Ángel Zamora,Precomisamiento zona 2.1 y zona 2.2,Codelco - Gasco RT,[Codelco - Gasco RT] Zona 2,I,22/09/2025,Gasco RT,"Día de trabajos administrativos, se realiza el...",3,Otro,N/A,N/A,Viaje de ida\nRetiro de materiales \nCompra de...
142,144,Ángel Zamora,Precom Zona 2.1,Codelco - Gasco RT,[Codelco - Gasco RT] Zona 2,I,23/09/2025,Gasco RT,Se realizan adicionales de cableado para resca...,3,Otro,Otro,N/A,Precomisamiento zona 2.1


In [6]:
#send_data(df_capas, 'Prueba_2', 'OTS')

In [29]:
df_capas.to_excel('resumen_trabajos.xlsx', index=False)

In [7]:
# update = sp.upload_file('https://graph.microsoft.com/v1.0/drives/b!_FsYntHkCECelI9711VeyMbyPx4aLChPm9Ub1jmcZ_XSssm7ywXaR6lOcXtuV77w/root:/Customer%20Experience%2FModelos%20de%20datos%2FModelo%20-%20%20Dashboard%20clientes/Prueba.xlsx:/workbook/tables/OTS/rows/add', df_capas)